# AI-Powered IT Ticket Creation & Categorization

Transformer-based NLP + Agent Architecture

## Dataset Download Instructions (Kaggle)

**Dataset:** IT Helpdesk Ticket Dataset  
https://www.kaggle.com/datasets/shivan118/it-helpdesk-ticket-dataset

Steps:
1. Login to Kaggle
2. Download the dataset
3. Extract CSV
4. Place CSV in same folder as this notebook


In [ ]:
# !pip install pandas transformers torch

In [ ]:
import re, json
import pandas as pd
from transformers import pipeline

In [ ]:
df = pd.read_csv('it_helpdesk_ticket.csv')
df.columns = df.columns.str.lower()
df = df[['description','category','priority']].dropna()
df.head()

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9 ]',' ',text)
    return text.strip()

In [ ]:
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')
CATEGORIES = df['category'].unique().tolist()

def classify_category(text):
    return classifier(text, CATEGORIES)['labels'][0]

In [ ]:
ner = pipeline('ner', model='dslim/bert-base-NER', aggregation_strategy='simple')

def extract_entities(text):
    ents = {}
    for e in ner(text):
        ents.setdefault(e['entity_group'], []).append(e['word'])
    return ents

In [ ]:
def infer_priority(text):
    t=text.lower()
    if any(w in t for w in ['down','error','crash','failed']): return 'High'
    if 'slow' in t: return 'Medium'
    return 'Low'

In [ ]:
def generate_ticket(text):
    return {
        'title': classify_category(text)+' Issue',
        'description': text,
        'category': classify_category(text),
        'priority': infer_priority(text),
        'entities': extract_entities(text)
    }

In [ ]:
sample = 'VPN error 809 on company laptop'
print(json.dumps(generate_ticket(sample), indent=4))